# Inspect phenology

Per-paddock phenology metrics — start / peak / end of season DOY,
amplitudes, integrals — answer questions like *which paddocks started
growing earliest this year?*, *how does my farm compare year-over-year?*,
or *are any paddocks behaving anomalously?*

> **Prerequisite.** A pipeline run covering **at least one full year**
> so phenology fitting is meaningful. The cell below uses
> `stub="stages_demo"` (full year 2024). For year-over-year comparison,
> use a multi-year run like 2022-01-01 → 2024-12-31.


In [ ]:
from datetime import date
from PaddockTS.query import Query

query = Query(
    bbox=[148.36265, -33.52606, 148.38265, -33.50606],
    start=date(2024, 1, 1),
    end=date(2024, 12, 31),
    stub="stages_demo",
)


## 1. Compute (or load) the phenology metrics

`estimate_phenology` returns `{year: pandas.DataFrame}`. Each row is one
paddock × one year. Re-running uses cached yearly time series — fast.


In [ ]:
from PaddockTS.Phenology.estimate_phenology import estimate_phenology

phen = estimate_phenology(query, variable="NDVI")
for year, df in phen.items():
    print(f"  {year}: {len(df)} paddocks")


## 2. Inspect one year's columns

`phenolopy` returns a wide table; the most useful metrics:

| Column | Meaning |
|---|---|
| `sos_times` / `sos_values` | Start of season DOY + index value |
| `pos_times` / `pos_values` | Peak of season DOY + value |
| `eos_times` / `eos_values` | End of season DOY + value |
| `aos_values` | Amplitude (peak − base) |
| `los_values` | Length of season in days |
| `sios_values` / `lios_values` | Small / long integral over the season |
| `num_peaks` | Independently-detected season count |


In [ ]:
year = list(phen)[-1]   # most recent year in the run
df = phen[year]
df[["paddock", "sos_times", "pos_times", "eos_times",
    "aos_values", "los_values", "num_peaks"]].head(10)


## 3. Distribution of SoS / PoS / EoS across paddocks


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, label in zip(axes,
                          ["sos_times", "pos_times", "eos_times"],
                          ["Start of season (DOY)",
                           "Peak of season (DOY)",
                           "End of season (DOY)"]):
    df[col].hist(bins=20, ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel(label)
    ax.set_ylabel("paddock count")
fig.suptitle(f"{year} phenology timing across {len(df)} paddocks")
plt.tight_layout()
plt.show()


## 4. Identify outliers

Paddocks in the top 10% latest SoS:


In [ ]:
late = df[df.sos_times > df.sos_times.quantile(0.9)].sort_values("sos_times", ascending=False)
late[["paddock", "sos_times", "pos_times", "eos_times", "aos_values"]]


## 5. Year-over-year comparison (if multiple years are present)

For a multi-year query, you can pivot SoS into a `(paddock × year)`
table and look at how each paddock's start-of-season shifted.


In [ ]:
import pandas as pd

if len(phen) >= 2:
    all_years = pd.concat([d.assign(year=y) for y, d in phen.items()])
    pivot = all_years.pivot(index="paddock", columns="year", values="sos_times")
    print("SoS DOY per paddock per year (first 5 paddocks):")
    print(pivot.head())

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.head(10).T.plot(ax=ax, marker="o", legend=False, alpha=0.5)
    ax.set_xlabel("Year")
    ax.set_ylabel("SoS (DOY)")
    ax.set_title("Start of season per paddock, year-over-year (first 10 paddocks)")
    plt.show()
else:
    print("Only one year of phenology results — re-run the pipeline over a")
    print("multi-year window to enable year-over-year comparison.")


## 6. Plot the seasonal curve for one outlier

Pick the paddock with the latest start-of-season and overlay its NDVI
trace with the SoS / PoS / EoS markers — useful to verify the
phenology fit makes sense.


In [ ]:
from PaddockTS.Phenology.make_yearly_paddock_time_series import make_yearly_paddock_time_series

yearly = make_yearly_paddock_time_series(query)
ds_year = yearly[year]

paddock_id = str(late.iloc[0].paddock)
row = late.iloc[0]

fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(ds_year.doy, ds_year.NDVI.sel(paddock=paddock_id),
           color="blue", s=24, label="NDVI")
ax.axvline(row.sos_times, color="green", linestyle="--", label="SoS")
ax.axvline(row.pos_times, color="blue",  linestyle="-.", label="PoS")
ax.axvline(row.eos_times, color="red",   linestyle=":",  label="EoS")
ax.set_xlabel("DOY")
ax.set_ylabel("NDVI")
ax.set_title(f"Paddock {paddock_id} — {year} (latest SoS in this run)")
ax.legend()
plt.tight_layout()
plt.show()


## See also

- [`PaddockTS.Plotting.phenology_plot`](https://thestochasticman.github.io/paddock-ts-local/api/plotting/#phenology-plot) — produces a multi-paddock × multi-year version of the curve above as a single PNG.
- [`06_inspect_time_series.ipynb`](06_inspect_time_series.ipynb) — drill into the underlying per-paddock × time cube.
